# Array-Sweep Playground

Interactive exploration of sweep figures — single-variable parameter
sweeps — from **SLURM job-array** runs that have been aggregated by
`scripts/aggregate_array_batch.py`.

**Why a separate notebook?** Array runs produce a per-experiment tree like:

```
results/exp_snr_sweep/
├── array_52671900_task_0/        per-task outputs (one per sweep slice)
├── array_52671900_task_1/
├── ...
├── array_52671900_aggregated/    ← merged result (this is what we load)
└── latest -> array_52671900_task_9/   (after rsync, often points at the wrong dir!)
```

The `latest` symlink is unreliable after `sync_results_from_hpc3.sh` — sync
order can land it on a per-task dir.  This notebook **never** uses `latest`
for array runs.  It picks the most recent `array_<jobid>_aggregated/`
explicitly (highest job ID), or you can pin a specific `ARRAY_ID`.

**Compatible experiments:** `snr_sweep`, `gamma_sweep`, `kappa_sweep`,
`clutter_cnr_sweep`, `n_ue_sweep`, `n_ap_sweep`, `antennas_sweep`.

In [ ]:
# Shared setup: make `cordis` importable from notebooks/ + IEEE rcParams.
import sys, logging
from pathlib import Path
from _playground_helpers import (
    setup_paper_style,
    load_aggregated_array_result, array_summary,
    print_per_task_summary, per_task_summary,
)
import numpy as np
import matplotlib.pyplot as plt

# Set use_latex=False if pdflatex isn't on PATH on this machine.
setup_paper_style(use_latex=True)
logging.basicConfig(level=logging.WARNING, format='%(levelname)-7s %(message)s')

## 1. Load aggregated array result

Set `exp_name` to the sweep parameter (`'snr'`, `'gamma'`, `'kappa'`,
`'clutter_cnr'`, `'n_ue'`, `'n_ap'`, `'antennas'`) and `ARRAY_ID` if you
want to pin a specific SLURM job ID.  Leave `ARRAY_ID = None` to grab the
most recent array (highest job ID under `results/exp_<name>_sweep/`).

In [ ]:
# ── The two knobs ──
exp_name  = 'snr'               # snr / gamma / kappa / clutter_cnr
                                # / n_ue / n_ap / antennas
ARRAY_ID  = None                # e.g. '52671900'; None = most recent

EXPERIMENT = f'{exp_name}_sweep'

result, array_id = load_aggregated_array_result(EXPERIMENT, array_id=ARRAY_ID)
assert result.kind == 'sweep', (
    f'This notebook is for sweep experiments; got kind={result.kind}.'
)
print(f'\nResolved ARRAY_ID = {array_id}')

## 2. Inspect aggregated metadata + per-task breakdown

`array_summary` prints the headline counts (trials, drops, realizations,
tasks, sweep axis) and the per-task table (task_id, seed, trials,
drops, realizations).

Seeds in `_array_common.sh` follow `SEED = BASE_SEED + ARRAY_TASK_ID`,
so they should form a contiguous range (e.g. `42-51` for a 10-task run
with the default `BASE_SEED=42`).  A gap signals a task that failed and
wasn't re-submitted.

In [ ]:
array_summary(result, EXPERIMENT, array_id)

Programmatic access to the same data via `per_task_summary` (returns
a list of dicts):

In [ ]:
rows = per_task_summary(EXPERIMENT, array_id)
print(f'{len(rows)} task records.')
if rows:
    print(f'First task: {rows[0]}')
total = sum(r['n_trials'] for r in rows)
print(f'Sum of per-task trials: {total}')

**Sweep-specific sanity check.**  Each axis value should have an entry
in `result.sweep_results` keyed by the value itself.  Confirm the
merged result has the expected number of sweep points:

In [ ]:
axis_values = list(result.sweep_results.keys())
print(f'Sweep axis:    {result.sweep_axis.name}')
print(f'Display:       {result.sweep_axis.display}')
print(f'Values ({len(axis_values)}):   {axis_values}')

# Per-axis-value algorithm count (should be identical across the sweep).
for v, sr in result.sweep_results.items():
    print(f'  axis={v}:  {len(sr.algorithm_results)} algorithms')

## 3. Quick sweep plot — one metric, all algorithms

`plot_sweep` takes the keyed-by-x results dict; its x-axis comes
from the dict's keys.  The `sweep_axis.display` metadata supplies
the human-readable axis label.

In [ ]:
from cordis.plotting import plot_sweep, figsize

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_sweep(result.sweep_results, metric='min_sinr_db', ax=ax)
ax.set_xlabel(result.sweep_axis.display)
ax.set_ylabel('min-SINR [dB]')
ax.set_title(f'{EXPERIMENT} (array {array_id}): '
             f'min-SINR vs. {result.sweep_axis.display}')
plt.show()

## 4. Side-by-side: SINR + SCNR

Most paper figures pair a communication metric (min-SINR) with a
sensing metric (mean-SCNR).  Build the two-panel layout inline.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=figsize(width='double', aspect=2.5/1.5))
plot_sweep(result.sweep_results, metric='min_sinr_db', ax=axes[0])
axes[0].set_xlabel(result.sweep_axis.display)
axes[0].set_ylabel('min-SINR [dB]')
axes[0].set_title('min-SINR [dB]')
plot_sweep(result.sweep_results, metric='mean_scnr_db', ax=axes[1])
axes[1].set_xlabel(result.sweep_axis.display)
axes[1].set_ylabel('mean-SCNR [dB]')
axes[1].set_title('mean-SCNR [dB]')
plt.tight_layout()
plt.show()

## 5. Algorithm filter — focus on CORDIS vs. Centralized

`plot_sweep` honors the same `only=` keyword as `plot_cdf`.

In [ ]:
PICK = ['CORDIS-Split', 'CORDIS-ADMM', 'Centralized']

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_sweep(result.sweep_results, metric='min_sinr_db', ax=ax, only=PICK)
ax.set_xlabel(result.sweep_axis.display)
ax.set_ylabel('min-SINR [dB]')
ax.set_title(f'{EXPERIMENT} (array {array_id}): CORDIS vs. Centralized')
plt.show()

## 6. Log-y for SCNR sweeps

Sensing metrics often span orders of magnitude across the sweep range.
Useful for `clutter_cnr_sweep` and `kappa_sweep` especially.

In [ ]:
fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_sweep(result.sweep_results, metric='mean_scnr_db', ax=ax, only=PICK)
ax.set_yscale('log')
ax.set_xlabel(result.sweep_axis.display)
ax.set_ylabel('mean SCNR (log scale)')
ax.set_title(f'{EXPERIMENT} (array {array_id}): mean-SCNR (log)')
plt.show()

## 7. Annotate the operating point

If your paper highlights a specific axis value (e.g. SNR=10 dB), draw
a vertical line + label at that point.

In [ ]:
# Set OPERATING_POINT to one of the sweep axis values.  None disables.
OPERATING_POINT = None       # e.g. 10  (for snr_sweep)

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_sweep(result.sweep_results, metric='min_sinr_db', ax=ax, only=PICK)
if OPERATING_POINT is not None and OPERATING_POINT in result.sweep_results:
    ax.axvline(OPERATING_POINT, color='k', linestyle=':',
               linewidth=0.8, alpha=0.6,
               label=f'{result.sweep_axis.display} = {OPERATING_POINT}')
    ax.legend()
ax.set_xlabel(result.sweep_axis.display)
ax.set_ylabel('min-SINR [dB]')
plt.show()

## 8. SINR floor (γ) + infeasibility annotation

`plot_sweep` (Stage 21 v3) supports the same γ-related kwargs as
`plot_cdf` — but their visual interpretation is adapted for the
sweep layout:

- **`gamma_db=<value>`**: draws a horizontal dashed line at
  `y = γ` (the target floor each algorithm tries to meet).  Most
  meaningful when the y-axis is `min_sinr_db`.
- **`show_feasible_only=True`**: at each sweep axis value,
  recompute the central tendency + error band using ONLY trials
  where `min_sinr_db ≥ γ`.  At axis values where every trial is
  infeasible the point is dropped, leaving a visible gap in the
  curve and a warning in the log.  Requires `gamma_db`.
- **`annotate_infeasibility=True`**: appends `(max inf=X.X%)`
  to each algorithm's legend label, reporting the **worst**
  infeasibility rate across all sweep axis values — the
  operating point where the algorithm struggles most.

The annotation auto-escapes `%` when `text.usetex=True`.

In [ ]:
# Full sweep, annotated with worst-case infeasibility rate.
GAMMA_DB = 5.0

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_sweep(
    result.sweep_results, metric='min_sinr_db', ax=ax, only=PICK,
    gamma_db=GAMMA_DB,
    show_feasible_only=False,        # all trials
    annotate_infeasibility=True,     # show (max inf=X.X%) in legend
)
ax.set_xlabel(result.sweep_axis.display)
ax.set_ylabel('min-SINR [dB]')
ax.set_title(rf'{EXPERIMENT}: sweep with $\gamma$={GAMMA_DB:.0f} dB floor')
plt.show()

In [ ]:
# Feasible-only: each (algo, axis-value) recomputed over the
# trials where that algo satisfied the γ floor.  Curve has a
# visible gap at any axis value where no trial was feasible.
fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_sweep(
    result.sweep_results, metric='min_sinr_db', ax=ax, only=PICK,
    gamma_db=GAMMA_DB,
    show_feasible_only=True,         # ← feasible trials only
    annotate_infeasibility=True,
)
ax.set_xlabel(result.sweep_axis.display)
ax.set_ylabel('min-SINR [dB] (feasible trials only)')
ax.set_title(rf'{EXPERIMENT}: feasible-only sweep at $\gamma$={GAMMA_DB:.0f} dB')
plt.show()

## 9. Save with provenance metadata

In [ ]:
from cordis.plotting import save_figure

out = save_figure(
    fig,
    base_path=f'../figures/playground/{EXPERIMENT}_array_{array_id}',
    formats=('pdf', 'png'),
    metadata={
        'Experiment': EXPERIMENT,
        'ArrayID':    str(array_id),
        'SweepAxis':  result.sweep_axis.name,
        'Notebook':   'playground_array_sweep',
    },
)
for p in out:
    print('wrote', p)

## 10. Compare multiple array runs (optional)

Set `ARRAY_A` / `ARRAY_B` to two array IDs to overlay their curves
(e.g. before vs. after a configuration change).  Leave as `None` to skip.

In [ ]:
ARRAY_A = None    # e.g. '52671900'
ARRAY_B = None    # e.g. '52680005'

if ARRAY_A and ARRAY_B:
    res_a, _ = load_aggregated_array_result(EXPERIMENT, array_id=ARRAY_A)
    res_b, _ = load_aggregated_array_result(EXPERIMENT, array_id=ARRAY_B)
    fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
    n_before = len(ax.lines)
    plot_sweep(res_a.sweep_results, metric='min_sinr_db', ax=ax, only=PICK)
    n_after_a = len(ax.lines)
    for ln in ax.lines[n_before:n_after_a]:
        ln.set_label(f'{ln.get_label()} (A={ARRAY_A})')
    plot_sweep(res_b.sweep_results, metric='min_sinr_db', ax=ax, only=PICK)
    for ln in ax.lines[n_after_a:]:
        ln.set_linestyle('--')
        ln.set_label(f'{ln.get_label()} (B={ARRAY_B})')
    ax.legend()
    ax.set_xlabel(res_a.sweep_axis.display)
    ax.set_ylabel('min-SINR [dB]')
    ax.set_title(f'{EXPERIMENT}: array {ARRAY_A} vs. {ARRAY_B}')
    plt.show()
else:
    print('Set ARRAY_A and ARRAY_B above to enable multi-array overlay.')